# Ускорение `trx` блока: техлид-подход через staging-таблицы

Идея:
- не читать большие `ods_alpha.scd1_trx*` в каждом расчете;
- один раз собрать узкий слой за февраль в `sandbox_ai`;
- дальше подключать в витрине уже агрегат `by n_agr`.


In [ ]:
import time
import pandas as pd
from rail_connectors.connection import connect

pd.options.display.max_columns = None
pd.options.display.width = None


In [ ]:
# Параметры
month_start = '2026-02-01'
month_end = (pd.to_datetime(month_start) + pd.offsets.MonthEnd(0)).strftime('%Y-%m-%d')

db = 'sandbox_ai'
tbl_keys = f'{db}.shestopalov_feb_base_keys'
tbl_trx_small = f'{db}.shestopalov_feb_trx_small'
tbl_trx_agg = f'{db}.shestopalov_feb_trx_agg_by_agr'

print('month_start =', month_start)
print('month_end =', month_end)
print('tbl_keys =', tbl_keys)
print('tbl_trx_small =', tbl_trx_small)
print('tbl_trx_agg =', tbl_trx_agg)


In [ ]:
imp = connect(
    to='IMPALA',
    extra_options={'db': db},
    driver_args={'tez.queue.name': 'ai'},
    kerberos={
        'keytab_path': '/home/jovyan/test_requests/tech.keytab',
        'use_credentials': True,
        'update_keytab': True,
    },
    user_params={'user_name': 'Shestopalov-VYur'}
)
imp._init_connection()

with imp:
    imp.execute('invalidate metadata ods_alpha.scd1_agreements')
    imp.execute('invalidate metadata ods_alpha.scd1_trx')
    imp.execute('invalidate metadata ods_alpha.scd1_trx_acq')
    imp.execute('invalidate metadata ods_alpha.scd1_trx_int')
print('Impala initialized')


## 1) Шаг ключей: создаем таблицу договорного периметра за февраль

Таблица содержит только ключи, которые нужны далее для trx-расчетов:
- `n_agr`, `agr_id_norm`, `n_cmp_client`, `month_start`, `month_end`.


In [ ]:
sql_create_keys = f"""
drop table if exists {tbl_keys};
create table {tbl_keys}
stored as parquet
as
select distinct
    cast(a.n_agr as string) as n_agr,
    cast(a.abs_agr_id as string) as agr_id_norm,
    cast(a.n_cmp_client as string) as n_cmp_client,
    cast('{month_start}' as date) as month_start,
    cast('{month_end}' as date) as month_end
from ods_alpha.scd1_agreements a
where a.acq_class = 'SA'
  and cast(a.d_valid_from as date) <= cast('{month_start}' as date)
  and (a.d_valid_to is null or cast(a.d_valid_to as date) >= cast('{month_start}' as date))
  and coalesce(a.ods_deleted_flg, '0') <> '1'
"""

t0 = time.time()
with imp:
    imp.execute(sql_create_keys)
    imp.execute(f'invalidate metadata {tbl_keys}')
elapsed = round(time.time() - t0, 2)
print('keys table created in', elapsed, 'sec')


In [ ]:
with imp:
    keys_stats = imp.fetch(f"""
        select
            count(*) as rows_cnt,
            count(distinct n_agr) as n_agr_cnt,
            count(distinct agr_id_norm) as agr_id_cnt,
            count(distinct n_cmp_client) as cmp_cnt
        from {tbl_keys}
    """)
display(keys_stats)


## 2) Шаг фактов: создаем узкий `trx_small` слой

Важно:
- сначала фильтр по дате в `scd1_trx`;
- затем join на `trx_acq` + ключи;
- затем добавляем `n_amt_fee` из `trx_int`.

Итоговая таблица держит только минимально нужные поля.


In [ ]:
sql_create_trx_small = f"""
drop table if exists {tbl_trx_small};
create table {tbl_trx_small}
stored as parquet
as
with trx_base as (
    select
        cast(t.n_trx as string) as n_trx,
        cast(t.n_amt_src as double) as n_amt_src
    from ods_alpha.scd1_trx t
    where to_date(cast(t.d_trx_orig as timestamp)) between cast('{month_start}' as date) and cast('{month_end}' as date)
      and t.c_nter is not null
      and coalesce(t.ods_deleted_flg, '0') <> '1'
),
trx_acq_small as (
    select
        cast(ta.n_trx as string) as n_trx,
        cast(ta.n_agr as string) as n_agr,
        coalesce(cast(ta.n_amt_tax as double), 0.0) as n_amt_tax
    from ods_alpha.scd1_trx_acq ta
    join {tbl_keys} k
      on cast(ta.n_agr as string) = k.n_agr
),
trx_join as (
    select
        ta.n_agr,
        ta.n_trx,
        tb.n_amt_src,
        ta.n_amt_tax
    from trx_base tb
    join trx_acq_small ta
      on ta.n_trx = tb.n_trx
)
select
    tj.n_agr,
    tj.n_trx,
    tj.n_amt_src,
    tj.n_amt_tax,
    coalesce(cast(i.n_amt_fee as double), 0.0) as n_amt_fee,
    cast('{month_start}' as date) as month_start
from trx_join tj
left join ods_alpha.scd1_trx_int i
  on cast(i.n_trx as string) = tj.n_trx
"""

t0 = time.time()
with imp:
    imp.execute(sql_create_trx_small)
    imp.execute(f'invalidate metadata {tbl_trx_small}')
elapsed = round(time.time() - t0, 2)
print('trx_small table created in', elapsed, 'sec')


In [ ]:
with imp:
    trx_small_stats = imp.fetch(f"""
        select
            count(*) as rows_cnt,
            count(distinct n_agr) as n_agr_cnt,
            count(distinct n_trx) as n_trx_cnt,
            sum(n_amt_src) as trx_sum,
            sum(n_amt_tax) as commission_from_ops_sum,
            sum(n_amt_fee) as int_component_sum
        from {tbl_trx_small}
    """)
display(trx_small_stats)


## 3) Шаг агрегата: создаем `trx_agg_by_agr`

Эту таблицу подключаем в других тетрадках вместо тяжелых join с `trx*`.


In [ ]:
sql_create_trx_agg = f"""
drop table if exists {tbl_trx_agg};
create table {tbl_trx_agg}
stored as parquet
as
select
    n_agr,
    cast('{month_start}' as date) as month_start,
    count(distinct n_trx) as trx_cnt,
    sum(n_amt_src) as trx_sum,
    sum(n_amt_tax) as commission_from_ops,
    sum(n_amt_fee) as int_component
from {tbl_trx_small}
group by n_agr
"""

t0 = time.time()
with imp:
    imp.execute(sql_create_trx_agg)
    imp.execute(f'invalidate metadata {tbl_trx_agg}')
elapsed = round(time.time() - t0, 2)
print('trx_agg table created in', elapsed, 'sec')


In [ ]:
with imp:
    trx_agg_stats = imp.fetch(f"""
        select
            count(*) as rows_cnt,
            count(distinct n_agr) as n_agr_cnt,
            sum(trx_cnt) as trx_cnt_sum,
            sum(trx_sum) as trx_sum,
            sum(commission_from_ops) as commission_from_ops_sum,
            sum(int_component) as int_component_sum
        from {tbl_trx_agg}
    """)
display(trx_agg_stats)

with imp:
    sample_agg = imp.fetch(f"""
        select *
        from {tbl_trx_agg}
        order by trx_sum desc
        limit 20
    """)
display(sample_agg)


## 4) Как использовать в основной витрине

Вместо тяжелого блока `trx*` в итоговом скрипте:
- join на `{tbl_trx_agg}` по `n_agr`;
- брать поля `trx_cnt`, `trx_sum`, `commission_from_ops`, `int_component`.

Так ты значительно сокращаешь память и время выполнения в финальной тетрадке.
